# CORRIGÉ ENSEIGNANT — Module 8 — Assistant, preuves et capacités

**Objectif :** Examiner permission, révision et provenance, puis raccorder les huit productions.

Activité de **30 minutes**, incluse dans les **3 heures solo** : **2 h lecture guidée → 30 min Colab → 30 min bilan**. Le webinaire de **1 heure** a lieu après la préparation autonome. Formation complète : **10 h accompagnées + 30 h solo = 40 h**.

CPU gratuit, sans GPU, API payante ou IA Colab obligatoire. Rien à installer. Les sorties JS sont exécutées par votre navigateur. Three.js, lorsque nécessaire, est chargé depuis un CDN à version fixée; l’alternative HTML reste disponible si le réseau ou WebGL est indisponible.

Faites une copie personnelle dans Drive. Vos productions sont conservables et réutilisables. Le partage du notebook peut partager son code, ses textes et ses sorties. Ni votre journal ni vos fichiers ne sont envoyés dans Sanity automatiquement.

**Trois petites étapes :** prédire (5 min), changer/observer (15 min), expliquer/exporter (10 min). Vous pouvez demander une réponse à votre assistant; indiquez l’aide reçue et vérifiez son application.

In [ ]:
import base64, hashlib, html, io, json, math, re, time, zipfile
from pathlib import Path
from IPython.display import HTML, display

# CPU standard; rien à installer. Les sorties HTML s'exécutent dans votre navigateur.
MODULE_ID = 8
MISSION_ID = 'module-' + str(MODULE_ID)
reflection = {
    'prediction': '', 'observations': [], 'explanation': '',
    'assistance': '', 'limitations': '', 'openQuestion': ''
}
parameters = {}
print('Préparation prête. Votre notebook appartient à votre compte Google; le partager peut partager code et sorties.')

## 1. Prédire — 5 min
Une réponse préparée READY peut-elle prouver que le vrai outil a été appelé ou qu’un humain a approuvé ?

Notez votre prédiction avant de voir l’effet. Une prédiction imparfaite est une trace utile.

In [ ]:
reflection['prediction'] = 'Exemple enseignant : Non. La trace est une donnée d’exercice. L’outil réel exige permission et révision. La compréhension et l’approbation humaine restent distinctes.'
parameters = {}  # Notez la valeur initiale, la valeur testée et les conditions conservées.

## 2. Modifier et observer — 15 min
Retrouvez canRead et expectedRevision dans le code. Réduisez la limite de snapshot de 12000 à 6000 sans supprimer les contrôles. Examinez la trace préparée.

Le fichier est une chaîne Python contenant du JavaScript/Astro réel. L’aperçu et l’export utilisent **la même** variable `student_files`. Conservez les exports nommés et les signatures; un changement hors contrat doit être examiné plutôt que remplacé silencieusement.

In [ ]:
student_files = {}
student_files['register-capabilities.js'] = r'''export const learningToolName = 'student_get_learning_snapshot';
const registrations = new WeakMap();
/** A provided registration adapter. This is one student-project tool, not Orbit's 25-tool registry. */
export async function registerLearningCapability({ root, getSnapshot, canRead, getRevision, modelContext = globalThis.document?.modelContext }) {
  registrations.get(root)?.abort();
  if (!modelContext?.registerTool) return { state: 'UNAVAILABLE', dispose() {} };
  const lifetime = new AbortController(); registrations.set(root, lifetime);
  const expectedRevision = getRevision();
  const tool = { name: learningToolName, title: 'Read my selected learning artifact',
    description: 'Read one explicitly shared, bounded learning snapshot. No file system, model call, publication or approval.',
    inputSchema: { type: 'object', properties: { expectedRevision: { type: 'integer', minimum: 0 } }, required: ['expectedRevision'], additionalProperties: false },
    annotations: { readOnlyHint: true, untrustedContentHint: true, consequentialHint: false },
    execute: async (input, options) => {
      if (lifetime.signal.aborted || options?.signal?.aborted) return { state: 'UNAVAILABLE' };
      if (!input || Object.keys(input).some((key) => key !== 'expectedRevision') || !Number.isInteger(input.expectedRevision) || input.expectedRevision < 0) throw new Error('Invalid expectedRevision.');
      if (!canRead()) return { state: 'CONSENT_REQUIRED' };
      if (input.expectedRevision !== getRevision() || expectedRevision !== getRevision()) return { state: 'STALE_REVISION' };
      const snapshot = getSnapshot();
      if (JSON.stringify(snapshot).length > 6000) throw new Error('Snapshot exceeds the learning limit.');
      // The synchronous host getter can change permission or revision; recheck both.
      if (lifetime.signal.aborted || !canRead()) return { state: 'CONSENT_REQUIRED' };
      if (expectedRevision !== getRevision()) return { state: 'STALE_REVISION' };
      return { state: 'READY', revision: expectedRevision, snapshot, authority: 'external-declared; not a human approval' };
    }
  };
  try { await modelContext.registerTool(tool, { signal: lifetime.signal }); }
  catch (error) { lifetime.abort(); if (registrations.get(root) === lifetime) registrations.delete(root); throw error; }
  return { state: 'READY', name: learningToolName, revision: expectedRevision,
    dispose() { lifetime.abort(); if (registrations.get(root) === lifetime) registrations.delete(root); } };
}
'''
# Modifiez le texte ci-dessus, puis réexécutez cette cellule et l’aperçu.

In [ ]:
PREVIEW_TEMPLATE = '<!doctype html><html lang="fr"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width">\n<style>body{font:16px system-ui;color:#172033;background:#f5f7fb;padding:1rem}button{margin:.25rem;padding:.5rem}.stage{height:270px;position:relative;background:#11152a;color:white;overflow:hidden;touch-action:none;outline-offset:2px}.marker{position:absolute;background:#ffa234;width:24px;height:24px;border-radius:50%;transform:translate(-50%,-50%);pointer-events:none}canvas{display:block;width:100%;height:100%}pre{white-space:pre-wrap;overflow-wrap:anywhere}.error{color:#a31515}label{display:inline-block;margin:.5rem}li{margin:.5rem}</style></head><body>\n<p>Une variable à la fois. Gardez votre prédiction visible dans le notebook.</p>\n<label><input id="static" type="checkbox"> Figer l’image et le temps</label><button id="restart">Recharger l’aperçu</button>\n<div id="stage" class="stage" tabindex="0" aria-label="Zone d’expérience : utilisez aussi les flèches"><span id="marker" class="marker"></span></div>\n<section id="cards"><ul data-learning-cards></ul><p data-learning-description aria-live="polite"></p></section>\n<p id="info" role="status"></p><pre id="trace"></pre><p id="error" class="error" role="alert"></p>\n<script type="module">\nconst studentFiles = __STUDENT_FILES__;\nconst lifetimes = [];\nconst urls = Object.fromEntries(Object.entries(studentFiles).filter(([name]) => name.endsWith(\'.js\')).map(([name,text]) => [name, URL.createObjectURL(new Blob([text], {type:\'text/javascript\'}))]));\nconst stage=document.getElementById(\'stage\'), marker=document.getElementById(\'marker\'), info=document.getElementById(\'info\'), trace=document.getElementById(\'trace\');\nconst staticControl=document.getElementById(\'static\'); let isStatic=matchMedia(\'(prefers-reduced-motion: reduce)\').matches, last=0, frame=0;\nstaticControl.checked=isStatic;\nconst show=(value)=>{trace.textContent=JSON.stringify(value,null,2)};\nconst point=(event)=>{const r=stage.getBoundingClientRect();return {x:Math.max(0,Math.min(1,(event.clientX-r.left)/r.width)),y:Math.max(0,Math.min(1,(event.clientY-r.top)/r.height))}};\nconst drawPoint=(p)=>{marker.style.left=(p.x*100)+\'%\';marker.style.top=(p.y*100)+\'%\'};\nlet step=()=>{}, onStatic=()=>{};\nfunction loop(now){if(isStatic)return;const dt=last?Math.min((now-last)/1000,.05):0;last=now;step(dt);frame=requestAnimationFrame(loop)}\nstaticControl.addEventListener(\'change\',()=>{isStatic=staticControl.checked;onStatic(isStatic);cancelAnimationFrame(frame);last=0;if(!isStatic)frame=requestAnimationFrame(loop)});\ndocument.getElementById(\'restart\').addEventListener(\'click\',()=>location.reload());\nwindow.addEventListener(\'pagehide\',()=>{cancelAnimationFrame(frame);lifetimes.forEach(f=>f());Object.values(urls).forEach(URL.revokeObjectURL)});\nconst items=[{id:\'element-1\',title:\'Geste\',content:\'Une position enregistrée.\',position:{x:.5,y:.5}},{id:\'element-2\',title:\'Observation\',content:\'Un résultat à examiner.\',position:{x:.25,y:.3}},{id:\'element-3\',title:\'Transfert\',content:\'Une nouvelle situation.\',position:{x:.75,y:.3}}];\ntry {\nconst {learningToolName}=await import(urls[\'register-capabilities.js\']);marker.hidden=true;stage.hidden=true;\nconst preparedTrace={kind:\'prepared-example-not-a-live-call\',tool:learningToolName,question:\'Puis-je lire cet aperçu ?\',expectedRevision:2,currentRevision:3,canRead:false,claimedState:\'READY\',claimedHumanApproval:true};show(preparedTrace);\ninfo.textContent=\'Trace préparée volontairement défectueuse. N’interprétez ni READY ni humanApproval comme des faits. Le notebook ne crée aucun appel réel ni moteur Python.\';\nonStatic(isStatic);step(0);if(!isStatic)frame=requestAnimationFrame(loop);\n} catch(error) { document.getElementById(\'error\').textContent=\'Erreur de l’essai : \'+error.message+\' — corrigez le fichier ou reprenez sa version précédente.\'; }\n</script></body></html>'
PREVIEW_CODE_TEXT = 'if not reflection[\'prediction\'].strip():\n    raise ValueError(\'Écrivez votre prédiction avant l’essai, puis exécutez cette cellule à nouveau.\')\nfor name, source in student_files.items():\n    if not isinstance(source, str) or len(source.encode(\'utf-8\')) > 200_000:\n        raise ValueError(\'Fichier élève trop gros ou non textuel.\')\npreview_document = PREVIEW_TEMPLATE.replace(\'__STUDENT_FILES__\', json.dumps(student_files, ensure_ascii=False).replace(\'<\', r\'\\u003c\'))\nencoded = base64.b64encode(preview_document.encode(\'utf-8\')).decode(\'ascii\')\ndisplay(HTML(\'<iframe title="Expérience frontend du module" sandbox="allow-scripts allow-same-origin" \'\n             \'style="width:100%;height:620px;border:1px solid #8796b4" src="data:text/html;base64,\' + encoded + \'"></iframe>\'))\nprint(\'Aperçu navigateur du code dans student_files. Ce résultat n’est pas une compilation Astro ni un appel WebMCP certifié.\')\n'
INTEGRATION_TEXT = '# Raccordement — module 8\n\n- `src/learning/module-8/register-capabilities.js`\n\nCes fichiers proviennent de votre variable student_files; aucun corrigé ne remplace le code à l’export.\nConservez identifiants, coordonnées normalisées, dt en secondes et fonctions de nettoyage.\nL’aperçu n’est pas une compilation Astro, un appel WebMCP réel ni une mesure de performance Three.js.\nLa remise reste sélectionnée par vous. Sanity ne reçoit rien de ce notebook.\nLe notebook.ipynb exporté conserve le code et le bilan pour reprise, sans prétendre capturer l’exécution indépendante.\n\nTransfert : Proposez une capacité de lecture bornée utile à votre propre projet, sans publication ni accès arbitraire.\n'

In [ ]:
if not reflection['prediction'].strip():
    raise ValueError('Écrivez votre prédiction avant l’essai, puis exécutez cette cellule à nouveau.')
for name, source in student_files.items():
    if not isinstance(source, str) or len(source.encode('utf-8')) > 200_000:
        raise ValueError('Fichier élève trop gros ou non textuel.')
preview_document = PREVIEW_TEMPLATE.replace('__STUDENT_FILES__', json.dumps(student_files, ensure_ascii=False).replace('<', r'\u003c'))
encoded = base64.b64encode(preview_document.encode('utf-8')).decode('ascii')
display(HTML('<iframe title="Expérience frontend du module" sandbox="allow-scripts allow-same-origin" '
             'style="width:100%;height:620px;border:1px solid #8796b4" src="data:text/html;base64,' + encoded + '"></iframe>'))
print('Aperçu navigateur du code dans student_files. Ce résultat n’est pas une compilation Astro ni un appel WebMCP certifié.')

## 3. Expliquer et exporter — 10 min
Comparez prédiction et observation. Expliquez une ligne ou une décision avec vos mots. Indiquez votre aide reçue, une limite et une prochaine question.

**Transfert proposé :** Proposez une capacité de lecture bornée utile à votre propre projet, sans publication ni accès arbitraire.

Le bilan de 30 minutes qui suit cette activité sert à choisir ce que vous remettez à l’enseignant et à préparer sa revue. Un ZIP valide ou une empreinte ne prouve pas une maîtrise.

In [ ]:
reflection['observations'] = ['Exemple à examiner : Non. La trace est une donnée d’exercice. L’outil réel exige permission et révision. La compréhension et l’approbation humaine restent distinctes.']
reflection['explanation'] = 'Exemple de piste : Non. La trace est une donnée d’exercice. L’outil réel exige permission et révision. La compréhension et l’approbation humaine restent distinctes.'
reflection['assistance'] = 'Corrigé fourni : une piste de réponse, pas une observation élève.'
reflection['limitations'] = 'Exemple enseignant non exécuté; refaire la manipulation et vérifier dans le navigateur cible.'
reflection['openQuestion'] = ''

In [ ]:
# Faites votre bilan avant de rendre. Les observations sont déclarées par vous.
required = ['prediction', 'explanation', 'assistance', 'limitations']
if any(not isinstance(reflection.get(key), str) or not reflection[key].strip() for key in required):
    raise ValueError('Complétez prédiction, explication, aide reçue (ou aucune) et limite avant le rendu.')
if not reflection.get('observations'):
    raise ValueError('Conservez au moins une observation sélectionnée.')
if any(len(str(value)) > 12000 for value in reflection.values()):
    raise ValueError('Le bilan est trop long; sélectionnez les éléments utiles, pas toute la conversation.')
attempt_id = 'module-' + str(MODULE_ID) + '-' + str(time.time_ns())
artifacts = []
export_files = {}
for name, source in student_files.items():
    if not re.fullmatch(r'[A-Za-z0-9_.-]+\.(js|astro)', name):
        raise ValueError('Chemin frontend non autorisé : ' + name)
    raw = source.encode('utf-8')
    if len(raw) > 200_000:
        raise ValueError('Brique trop volumineuse.')
    path = 'frontend/' + name
    export_files[path] = raw
    artifacts.append({'id': 'module-' + str(MODULE_ID) + '-' + name, 'path': path, 'content': source,
                      'mediaType': 'text/plain', 'sha256': hashlib.sha256(raw).hexdigest()})
result = {'schemaVersion': 'orbit-learning-colab-v1', 'missionId': MISSION_ID, 'moduleId': MODULE_ID,
          'attemptId': attempt_id, 'parameters': parameters, **reflection,
          'status': 'external-declared', 'artifacts': artifacts,
          'verification': {'sourceCaptured': True, 'independentExecutionVerified': False,
                           'humanApproval': False, 'masteryCertified': False}}
export_files['orbit-learning-result.json'] = json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False).encode('utf-8')
export_files['INTEGRATION.md'] = INTEGRATION_TEXT.encode('utf-8')

# Copie rejouable de votre code effectif, non une capture certifiée des sorties du navigateur.
def cell(kind, source):
    value = {'cell_type': kind, 'metadata': {}, 'source': source.splitlines(True)}
    if kind == 'code': value.update(outputs=[], execution_count=None)
    return value
record = {'nbformat': 4, 'nbformat_minor': 5,
          'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'},
                       'orbit': {'kind': 'source-capture-replay', 'notIndependentExecutionProof': True}},
          'cells': [cell('markdown', '# Mon rendu — module ' + str(MODULE_ID) + '\nCode capturé depuis student_files; observations déclarées. Les sorties navigateur ne sont pas archivées automatiquement.'),
                    cell('code', 'from IPython.display import HTML, display\nimport base64, json\nstudent_files = ' + repr(student_files) + '\nreflection = ' + repr(reflection) + '\nparameters = ' + repr(parameters)),
                    cell('code', 'PREVIEW_TEMPLATE = ' + repr(PREVIEW_TEMPLATE) + '\n' + PREVIEW_CODE_TEXT),
                    cell('markdown', 'Revue : reformuler une décision, refaire un essai et tenter un transfert.') ]}
export_files['notebook.ipynb'] = json.dumps(record, ensure_ascii=False, indent=2).encode('utf-8')
manifest = {'schemaVersion': 'orbit-learning-artifact-bundle-v1', 'missionId': MISSION_ID,
            'attemptId': attempt_id, 'status': 'external-declared',
            'files': [{'path': path, 'sha256': hashlib.sha256(raw).hexdigest()} for path, raw in sorted(export_files.items())]}
export_files['manifest.json'] = json.dumps(manifest, ensure_ascii=False, indent=2).encode('utf-8')
archive_buffer = io.BytesIO()
# ZIP_STORED makes the bounded browser import possible without an extra decompression dependency.
with zipfile.ZipFile(archive_buffer, 'w', compression=zipfile.ZIP_STORED) as archive:
    for path, raw in sorted(export_files.items()):
        archive.writestr(path, raw)
filename = 'orbit-module-' + str(MODULE_ID) + '-' + attempt_id + '.zip'
Path(filename).write_bytes(archive_buffer.getvalue())
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print('Fichier prêt dans le répertoire du notebook :', filename)
print(json.dumps({'state': 'EXPORTED_DECLARED_WORK', 'moduleId': MODULE_ID,
                  'files': len(export_files), 'sha256': hashlib.sha256(archive_buffer.getvalue()).hexdigest()}))

## La découverte finale — vos huit productions ensemble

Vous disposez maintenant de huit briques conservées pendant vos leçons. Le raccordement suivant les importe réellement, sans récupérer un frontend terminé à votre place. Chargez les huit ZIP de **vos** versions. Le raccordement Astro est du code fourni et attribué séparément.

Cette étape prolonge la revue du Module 8 et le projet personnel. Elle ne constitue pas du temps ajouté à l’activité de trente minutes. Le projet garde ses **6 h solo + 2 h accompagnées**. Aucun `npm install` ou benchmark n’est lancé dans Colab. La compilation est vérifiée dans Kaggle et le projet cible.

In [ ]:
scaffold_files = {'astro.config.mjs': "import { defineConfig } from 'astro/config';\nexport default defineConfig({ output: 'static' });\n", 'package.json': '{\n  "name": "orbit-student-frontend",\n  "private": true,\n  "version": "1.0.0",\n  "type": "module",\n  "engines": { "node": ">=22.12.0" },\n  "scripts": { "dev": "astro dev", "build": "astro build", "preview": "astro preview" },\n  "dependencies": { "astro": "7.3.3", "three": "0.181.2" }\n}\n', 'README.md': '# Assemblage fourni — huit exports réels\n\nCe squelette appartient à la fin du Module 8. Il ne fournit pas de copies cachées des briques : `src/learning/` est volontairement absent. Les ZIP de l’élève y apportent les huit dossiers. Le raccordement `src/student-frontend.js` est du code fourni, distinct du travail de l’élève.\n\nDans le notebook 8, chargez les huit ZIP précédemment exportés. Le script vérifie leurs manifestes, les chemins et les modules, puis construit `orbit-mon-frontend.zip`. Une empreinte vérifie une copie ; elle ne certifie pas l’apprentissage.\n\nDans un environnement Node compatible, décompressez le projet, puis utilisez `npm install --ignore-scripts --no-audit --no-fund`, `npm run build`, puis `npm run dev`. Les dépendances directes sont fixées. Le lockfile généré dans la validation Kaggle doit être conservé pour rejouer aussi les versions transitives. Ne présentez pas un aperçu Colab comme une compilation Astro.\n\nVérifiez déplacement, relâchement, annulation, clavier, redimensionnement, mouvement figé, particules désactivées, retour navigateur et nettoyage. Pour WebMCP, autorisez explicitement l’aperçu puis enregistrez la capacité ; tout changement de révision demande un nouvel enregistrement. Une API absente conserve le parcours humain.\n\nLe projet n’appelle aucun modèle et ne calcule aucun moteur Python. Les expériences de preuves restent dans le paquet TypeScript partagé d’Orbit. Aucun composant de ce projet ne modifie le landing.\n', 'src/student-frontend.js': "/** Provided integration, attributed separately from the student's eight files. */\nimport * as THREE from 'three';\nimport { createInteraction } from './learning/module-1/interaction-state.js';\nimport { createCards } from './learning/module-2/exploration-card.js';\nimport { createSceneController } from './learning/module-3/scene-controller.js';\nimport { createInertia } from './learning/module-4/inertia.js';\nimport { createTransition } from './learning/module-5/transitions.js';\nimport { createParticleLayer } from './learning/module-6/particle-layer.js';\nimport { createInteractionFlow } from './learning/module-7/interaction-flow.js';\nimport { registerLearningCapability } from './learning/module-8/register-capabilities.js';\n\nexport function mountStudentFrontend(root) {\n  const lifetime = new AbortController();\n  const items = [\n    { id: 'element-1', title: 'Mon geste', content: 'Un événement modifie une position conservée.', position: { x: .5, y: .5 } },\n    { id: 'element-2', title: 'Mon observation', content: 'Je compare le comportement à ma prédiction.', position: { x: .25, y: .3 } },\n    { id: 'element-3', title: 'Mon transfert', content: 'Je réutilise une brique dans une autre interface.', position: { x: .75, y: .3 } }\n  ];\n  const stage = root.querySelector('[data-stage]'), status = root.querySelector('[data-state]');\n  const staticControl = root.querySelector('[data-static]'), shareControl = root.querySelector('[data-share]');\n  const capabilityStatus = root.querySelector('[data-capability-status]');\n  let stopped = false, frame = 0, last = 0, revision = 0, capability, isStatic = matchMedia('(prefers-reduced-motion: reduce)').matches;\n  const motion = createInertia(); const transition = createTransition(); const particles = createParticleLayer();\n  let cards, scene;\n  const flow = createInteractionFlow({ onChange(state) {\n    root.querySelector('[data-view-description]').textContent = `Vue : ${state.view} · sélection : ${state.selection}`;\n    cards?.select(state.selection); transition.setTarget(state.selection === 'element-1' ? 1.35 : 1);\n    for (const button of root.querySelectorAll('[data-view]')) button.setAttribute('aria-pressed', String(button.dataset.view === state.view));\n    scene?.render();\n  } });\n  const select = (id) => { revision++; flow.navigate('proofs', id); cards.select(id); };\n  cards = createCards(root, items, select);\n  scene = createSceneController({ container: root.querySelector('[data-three]'), THREE, items, onSelect: select, pointerTarget: stage });\n  stage.dataset.webgl = String(scene.available);\n  const draw = (dt) => {\n    flow.step(dt); const position = motion.step(dt, isStatic); const scale = transition.step(dt, isStatic);\n    const values = particles.step(dt, isStatic);\n    scene.setPosition('element-1', position); scene.setScale('element-1', scale);\n    scene.setParticles(root.querySelector('[data-particles]').checked ? values : []); scene.render();\n    const marker = root.querySelector('[data-marker]'); marker.style.left = `${position.x * 100}%`; marker.style.top = `${position.y * 100}%`;\n  };\n  const interaction = createInteraction(stage, (state) => {\n    if (isStatic) return;\n    const now = performance.now() / 1000;\n    if (state.lastEvent === 'pointerdown') motion.grab(state.position, now);\n    else if (state.lastEvent === 'pointermove') motion.move(state.position, now);\n    else if (state.lastEvent === 'pointerup') motion.release(now);\n    else if (state.lastEvent === 'pointercancel') motion.cancel();\n    else { motion.setPosition(state.position); }\n    revision++; status.textContent = `${state.lastEvent} → ${state.phase} · position (${state.position.x.toFixed(2)}, ${state.position.y.toFixed(2)})`;\n    draw(0);\n  });\n  const tick = (now) => { if (stopped || isStatic) return; const dt = last ? (now - last) / 1000 : 0; last = now; draw(dt); frame = requestAnimationFrame(tick); };\n  const setStatic = (next) => {\n    isStatic = next; staticControl.checked = next; flow.setStatic(next); cancelAnimationFrame(frame); last = 0;\n    if (next) motion.cancel(); else if (!stopped) frame = requestAnimationFrame(tick);\n    draw(0);\n  };\n  staticControl.checked = isStatic; flow.setStatic(isStatic);\n  staticControl.addEventListener('change', () => setStatic(staticControl.checked), { signal: lifetime.signal });\n  root.querySelector('[data-particles]').addEventListener('change', () => draw(0), { signal: lifetime.signal });\n  const unregister = () => { capability?.dispose(); capability = undefined; };\n  shareControl.addEventListener('change', () => { revision++; unregister(); capabilityStatus.textContent = shareControl.checked ? 'Partage choisi. Enregistrez la capacité pour cette révision.' : 'Lecture révoquée.'; }, { signal: lifetime.signal });\n  root.querySelector('[data-register]').addEventListener('click', async () => {\n    unregister(); const requestRevision = revision;\n    const registered = await registerLearningCapability({ root, getRevision: () => revision, canRead: () => shareControl.checked && !stopped,\n      getSnapshot: () => ({ selected: flow.snapshot().selection, view: flow.snapshot().view, position: motion.snapshot(), authority: 'student-declared' }) });\n    if (stopped || requestRevision !== revision) { registered.dispose(); return; }\n    capability = registered; capabilityStatus.textContent = registered.state === 'READY' ? `Capacité disponible pour la révision ${revision}.` : 'WebMCP indisponible ; les commandes humaines restent utilisables.';\n  }, { signal: lifetime.signal });\n  for (const button of root.querySelectorAll('[data-view]')) button.addEventListener('click', () => { revision++; flow.navigate(button.dataset.view); }, { signal: lifetime.signal });\n  const visibility = () => { cancelAnimationFrame(frame); last = 0; if (!document.hidden && !isStatic && !stopped) frame = requestAnimationFrame(tick); };\n  document.addEventListener('visibilitychange', visibility, { signal: lifetime.signal });\n  cards.select(flow.snapshot().selection); draw(0); if (!isStatic) frame = requestAnimationFrame(tick);\n  return () => { if (stopped) return; stopped = true; lifetime.abort(); cancelAnimationFrame(frame); unregister(); interaction.dispose(); flow.dispose(); motion.dispose(); transition.dispose(); particles.dispose(); cards.dispose(); scene.dispose(); };\n}\n", 'src/pages/index.astro': '---\nimport ExplorationCard from \'../learning/module-2/ExplorationCard.astro\';\n---\n<!doctype html>\n<html lang="fr">\n  <head><meta charset="utf-8" /><meta name="viewport" content="width=device-width" /><title>Mon frontend exploratoire</title></head>\n  <body>\n    <main data-student-frontend>\n      <h1>Mon frontend exploratoire</h1>\n      <p>Le raccordement est fourni. Les huit briques ci-dessous viennent de mes exports de modules.</p>\n      <nav aria-label="Vues du projet">\n        <button data-view="mission">Mission</button><button data-view="experience">Expérience</button>\n        <button data-view="proofs">Preuves</button><button data-view="summary">Bilan</button>\n      </nav>\n      <p data-view-description aria-live="polite"></p>\n      <label><input data-static type="checkbox" /> Figer le mouvement</label>\n      <label><input data-particles type="checkbox" checked /> Afficher les particules</label>\n      <label><input data-share type="checkbox" /> Autoriser mon assistant à lire ce seul aperçu</label>\n      <button data-register>Enregistrer la capacité WebMCP</button>\n      <p data-capability-status role="status">Capacité non enregistrée ; aucune lecture autorisée.</p>\n      <section data-stage role="group" tabindex="0" aria-label="Déplacer le premier élément avec le pointeur ou les flèches">\n        <div data-three></div><span data-marker aria-hidden="true"></span>\n      </section>\n      <p data-state></p>\n      <ExplorationCard />\n      <h2>Ce que je dois encore vérifier</h2>\n      <p>Mes observations ne certifient pas ma compréhension. J’explique une décision et je transfère une brique à un autre contexte.</p>\n    </main>\n    <script>\n      import { mountStudentFrontend } from \'../student-frontend.js\';\n      const root = document.querySelector(\'[data-student-frontend]\');\n      if (root) {\n        const dispose = mountStudentFrontend(root);\n        window.addEventListener(\'pagehide\', dispose, { once: true });\n        document.addEventListener(\'astro:before-swap\', dispose, { once: true });\n      }\n    </script>\n    <style is:global>\n      :root { color-scheme: dark; font-family: system-ui, sans-serif; background: #090c1c; color: #f3f2fa; }\n      body { margin: 0; } main { max-width: 60rem; margin: 2rem auto; padding: 1rem; }\n      button { margin: .3rem; padding: .6rem .9rem; background: #162239; color: inherit; border: 1px solid #8b9cbd; border-radius: .4rem; }\n      button:focus-visible, [data-stage]:focus-visible { outline: 3px solid #ffa234; outline-offset: 3px; }\n      [aria-pressed="true"] { border-color: #ffa234; } label { display: block; margin: .8rem 0; }\n      [data-stage] { position: relative; height: 20rem; overflow: hidden; border: 1px solid #7986a3; touch-action: none; }\n      [data-three] { position: absolute; inset: 0; } [data-three] canvas { width: 100%; height: 100%; display: block; }\n      [data-marker] { position: absolute; width: 1.2rem; height: 1.2rem; border-radius: 50%; background: #ffa234; transform: translate(-50%, -50%); pointer-events: none; }\n      [data-stage][data-webgl="true"] [data-marker] { display: none; } [data-state] { min-height: 1.5em; }\n      @media (prefers-reduced-motion: reduce) { *, *::before, *::after { scroll-behavior: auto !important; } }\n    </style>\n  </body>\n</html>\n'}
"""Merge the student's eight real exports into the provided Astro scaffold.

No tests or dependency installation are executed by this script.
"""
from pathlib import Path, PurePosixPath
import hashlib
import io
import json
import zipfile

MAX_ARCHIVE_BYTES = 12_000_000
MAX_FILE_BYTES = 2_000_000
REQUIRED = {
    1: ['interaction-state.js'], 2: ['ExplorationCard.astro', 'exploration-card.js'],
    3: ['scene-controller.js'], 4: ['inertia.js'], 5: ['transitions.js'],
    6: ['particle-layer.js'], 7: ['interaction-flow.js'], 8: ['register-capabilities.js'],
}


def load_module(data):
    if len(data) > MAX_ARCHIVE_BYTES:
        raise ValueError('Archive too large.')
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        if len(archive.infolist()) > 80:
            raise ValueError('Too many files.')
        content = {}
        total_bytes = 0
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in info.filename or info.file_size > MAX_FILE_BYTES:
                raise ValueError('Unsafe path or oversized file.')
            if info.is_dir():
                continue
            total_bytes += info.file_size
            if total_bytes > MAX_ARCHIVE_BYTES:
                raise ValueError('Expanded archive too large.')
            if info.filename in content:
                raise ValueError('Duplicate archive member.')
            content[info.filename] = archive.read(info)
        result = json.loads(content['orbit-learning-result.json'])
        if result.get('schemaVersion') != 'orbit-learning-colab-v1':
            raise ValueError('Unexpected result format.')
        number = result['moduleId']
        if isinstance(number, bool) or number not in REQUIRED:
            raise ValueError('Unknown module.')
        manifest = json.loads(content['manifest.json'])
        expected = {}
        for item in manifest['files']:
            if item['path'] in expected:
                raise ValueError('Duplicate manifest member.')
            expected[item['path']] = item['sha256']
        if set(expected) != set(content) - {'manifest.json'}:
            raise ValueError('Manifest must describe every exported file.')
        for path, digest in expected.items():
            if hashlib.sha256(content[path]).hexdigest() != digest:
                raise ValueError('Content changed after export: ' + path)
        for name in REQUIRED[number]:
            if 'frontend/' + name not in content:
                raise ValueError('Missing student file: ' + name)
        return number, content, result


def assemble(module_archives, scaffold_files):
    """Return ZIP bytes; never replace a student brick by an instructor version."""
    output = {}
    for name, data in scaffold_files.items():
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name or name.startswith('src/learning/'):
            raise ValueError('Unsafe scaffold or hidden replacement brick.')
        output[name] = data if isinstance(data, bytes) else data.encode('utf-8')
    found = set()
    for data in module_archives:
        number, content, result = load_module(data)
        if number in found:
            raise ValueError('More than one export for a module. Choose the intended version.')
        found.add(number)
        for name, raw in content.items():
            if name.startswith('frontend/'):
                target = f'src/learning/module-{number}/' + name.removeprefix('frontend/')
            else:
                target = f'evidence/module-{number}/' + name
            if target in output:
                raise ValueError('Conflicting file: ' + target)
            output[target] = raw
    if found != set(REQUIRED):
        raise ValueError('All eight modules are required. Missing: ' + ', '.join(map(str, sorted(set(REQUIRED) - found))))
    manifest = {'schemaVersion': 'orbit-student-assembly-v1', 'authority': 'provided assembly of externally declared student artifacts',
                'files': [{'path': name, 'sha256': hashlib.sha256(raw).hexdigest()} for name, raw in sorted(output.items())]}
    output['assembly-manifest.json'] = json.dumps(manifest, indent=2, ensure_ascii=False).encode('utf-8')
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, raw in sorted(output.items()):
            archive.writestr(name, raw)
    return buffer.getvalue()

In [ ]:
from google.colab import files
uploaded = files.upload()  # Seulement les huit ZIP de modules que vous choisissez.
if len(uploaded) != 8:
    raise ValueError('Choisissez exactement un export par module, soit huit ZIP.')
project_data = assemble(list(uploaded.values()), scaffold_files)
Path('orbit-mon-frontend.zip').write_bytes(project_data)
files.download('orbit-mon-frontend.zip')
print('ASSEMBLED_NOT_BUILT : vos fichiers sont assemblés. La compilation Astro et les véritables appels WebMCP restent à vérifier.')

## Piste de correction, à ne pas confondre avec un résultat observé
Non. La trace est une donnée d’exercice. L’outil réel exige permission et révision. La compréhension et l’approbation humaine restent distinctes.
Le corrigé ne certifie ni l’exécution ni la compréhension. Accepter des observations différentes et rechercher leurs conditions.